In [0]:
%pip install pymupdf==1.26.1


how to get page num:

1. put all #header into a list
2. Extract pdf again and taking into consideration every page 
3. Find the page of location of first element in the list and the one after that 
4. Instert that into the metadata as Page: first_page-last-page
5. Do this for every element in the list
6. Last element will be done by fiding its element and the page length of document

In [0]:
%pip install dotenv
%pip install langchain_community
%pip install langchain_huggingface
%pip install langchain-openai==0.3.21
%pip install git+https://github.com/huggingface/transformers.git
%pip install faiss-cpu
%pip install rank_bm25
%pip install docling
%pip install PyPDF2
%pip install markdown

In [0]:
from PyPDF2 import PdfReader

def extract_text_from_pdf_pypdf2(pdf_path):
    """
    Extracts text from each page of a PDF using PyPDF2.

    Args:
        pdf_path: The path to the PDF file.
    """
    with open(pdf_path, 'rb') as file:
        pdf_reader = PdfReader(file)
        num_pages = len(pdf_reader.pages)
        print(f"The PDF has {num_pages} pages.")

        for page_num in range(num_pages):
            page = pdf_reader.pages[page_num]
            text = page.extract_text()
            print(f"--- Content from Page {page_num + 1} ---")
            print(text)
            print("-" * 20)

# Example usage:
extract_text_from_pdf_pypdf2('29173_h00.pdf')

In [0]:
%restart_python

In [0]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_community.vectorstores import FAISS
from typing import List
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.pipeline.vlm_pipeline import VlmPipeline
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_openai import ChatOpenAI
import logging
import time
from pathlib import Path
from docling.chunking import HierarchicalChunker
import pandas as pd
from docling_core.types.doc import ImageRefMode, PictureItem
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

In [0]:

IMAGE_RESOLUTION_SCALE = 2.0
pdf = "29173_h00.pdf"

pipeline_options = PdfPipelineOptions()
pipeline_options.images_scale = IMAGE_RESOLUTION_SCALE
pipeline_options.generate_page_images = True
pipeline_options.generate_picture_images = True

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)
conv_res = doc_converter.convert(pdf)

In [0]:
from docling_core.types.doc import  TextItem
c=0
for element, i in conv_res.document.iterate_items():
    if isinstance(element, TextItem):
        if "page no" in element.text:
            print(f"page is {element.text}")
            break
    c+=1
    
    

In [0]:

IMAGE_RESOLUTION_SCALE = 2.0
pdf = "29173_h00.pdf"
def main():
    data_folder = Path("").parent / ""
    input_doc_path = data_folder / pdf
    output_dir_images = Path("scratch")
    output_dir_tables = Path("scratch")

    # Setup pipeline options for images
    pipeline_options = PdfPipelineOptions()
    pipeline_options.images_scale = IMAGE_RESOLUTION_SCALE
    pipeline_options.generate_page_images = True
    pipeline_options.generate_picture_images = True

    doc_converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

    start_time = time.time()

    conv_res = doc_converter.convert(input_doc_path)

    # Create output directories
    output_dir_images.mkdir(parents=True, exist_ok=True)
    output_dir_tables.mkdir(parents=True, exist_ok=True)

    doc_filename = conv_res.input.file.stem

    # Save images of figures
    picture_counter = 0
    
    for element, _level in conv_res.document.iterate_items():
        
        if isinstance(element, PictureItem):
            picture_counter += 1
            element_image_filename = (
                output_dir_images / f"{doc_filename}-picture-{picture_counter}.png"
            )
            with element_image_filename.open("wb") as fp:
                element.get_image(conv_res.document).save(fp, "PNG")

    # Save markdown with embedded pictures
    md_filename = output_dir_images / f"{doc_filename}-with-images.md"
    conv_res.document.save_as_markdown(md_filename, image_mode=ImageRefMode.EMBEDDED)

    # Save markdown with externally referenced pictures
    md_filename = output_dir_images / f"{doc_filename}-with-image-refs.md"
    conv_res.document.save_as_markdown(md_filename, image_mode=ImageRefMode.REFERENCED)

    # # Save HTML with externally referenced pictures
    # html_filename = output_dir_images / f"{doc_filename}-with-image-refs.html"
    # conv_res.document.save_as_html(html_filename, image_mode=ImageRefMode.REFERENCED)

    # Export tables
    for table_ix, table in enumerate(conv_res.document.tables):
        table_df: pd.DataFrame = table.export_to_dataframe()
        print(f"## Table {table_ix}")
        print(table_df.to_markdown())

        # Save the table as csv
        element_csv_filename = output_dir_tables / f"{doc_filename}-table-{table_ix + 1}.csv"
        #_log.info(f"Saving CSV table to {element_csv_filename}")
        table_df.to_csv(element_csv_filename)

        # # Save the table as html
        # element_html_filename = output_dir_tables / f"{doc_filename}-table-{table_ix + 1}.html"
        # _log.info(f"Saving HTML table to {element_html_filename}")
        # with element_html_filename.open("w") as fp:
        #     fp.write(table.export_to_html(doc=conv_res.document))

if __name__ == "__main__":
    main()

In [0]:
# # works

# import json
# import base64
# import mimetypes
# from openai import OpenAI
# from pathlib import Path

# try:
#     os.environ['OPENAI_API_KEY'] = dbutils.secrets.get(scope='common', key="NetDoc_OPENAI_KEY")
# except NameError:
#     raise EnvironmentError("dbutils is not available. Ensure you're running this in a supported environment or provide the OpenAI API key manually.")


# # Initialize the OpenAI client
# # Ensure your OPENAI_API_KEY environment variable is set
# client = OpenAI()

# def get_image_description_openai(image_path: Path) -> str | None:
#     """
#     Generates a description for a local image file using OpenAI's vision model.

#     Args:
#         image_path (Path): The path to the local image file.

#     Returns:
#         str | None: The generated description, or None if an error occurs.
#     """
#     if not image_path.exists() or not image_path.is_file():
#         print(f"Error: Image file not found at {image_path}")
#         return None

#     try:
#         # Read and Base64 encode the image
#         with open(image_path, "rb") as image_file:
#             base64_image = base64.b64encode(image_file.read()).decode('utf-8')

#         # Determine the MIME type
#         mime_type, _ = mimetypes.guess_type(image_path)
#         if not mime_type:
#             # Fallback or raise error if MIME type can't be determined
#             # For common research paper images, png or jpeg are likely
#             if image_path.suffix.lower() == ".png":
#                 mime_type = "image/png"
#             elif image_path.suffix.lower() in [".jpg", ".jpeg"]:
#                 mime_type = "image/jpeg"
#             else:
#                 print(f"Error: Could not determine MIME type for {image_path}")
#                 return None
        
#         image_data_url = f"data:{mime_type};base64,{base64_image}"

#         system_prompt = (
#             "You are an expert research assistant. Your task is to describe scientific figures "
#             "extracted from research papers. Focus on the key information presented in the figure. "
#             "Describe what the figure depicts (e.g., a graph, a diagram, a micrograph). "
#             "If it's a graph, mention the axes and the general trend or comparison shown. "
#             "If it's a diagram, explain what process or system it illustrates. "
#             "Be concise and informative, as if you were writing a caption for this figure."
#         )
        
#         user_prompt_message = f"Please describe the content of this scientific figure."

#         response = client.chat.completions.create(
#             model="gpt-4o-mini", # Or "gpt-4-turbo" or other vision-capable models
#             messages=[
#                 {
#                     "role": "system",
#                     "content": system_prompt
#                 },
#                 {
#                     "role": "user",
#                     "content": [
#                         {"type": "text", "text": user_prompt_message},
#                         {
#                             "type": "image_url",
#                             "image_url": {
#                                 "url": image_data_url,
#                             }
#                         },
#                     ],
#                 }
#             ],
#             max_tokens=500, # Adjust as needed
#             temperature=0.3 # Adjust for more factual vs. creative descriptions
#         )
#         return response.choices[0].message.content

#     except Exception as e:
#         print(f"An error occurred while generating image description for {image_path}: {e}")
#         return None



In [0]:
%pip install -qU pip docling transformers

In [0]:
import re
import os
import base64
import mimetypes
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

im_list = []

# --- OpenAI Client Setup ---
try:
    os.environ['OPENAI_API_KEY'] = dbutils.secrets.get(scope='common', key="NetDoc_OPENAI_KEY")
except NameError:
    raise EnvironmentError("dbutils is not available. Ensure you're running this in a supported environment or provide the OpenAI API key manually.")


client = OpenAI()


# --- Your Image Description Function ---
# (This function is correct and does not need changes)
def get_image_description_openai(image_path: Path) -> str | None:
    """
    Generates a description for a local image file using OpenAI's vision model.
    """
    if not image_path.exists() or not image_path.is_file():
        # This error is what you were seeing.
        print(f"Error: Image file not found at {image_path}")
        return None
    try:
        with open(image_path, "rb") as image_file:
            base64_image = base64.b64encode(image_file.read()).decode('utf-8')
        mime_type, _ = mimetypes.guess_type(image_path)
        if not mime_type:
            if image_path.suffix.lower() == ".png": mime_type = "image/png"
            elif image_path.suffix.lower() in [".jpg", ".jpeg"]: mime_type = "image/jpeg"
            else:
                print(f"Error: Could not determine MIME type for {image_path}")
                return None
        image_data_url = f"data:{mime_type};base64,{base64_image}"
        system_prompt = (
            "You are an expert research assistant. Your task is to describe scientific figures "
            "extracted from research papers. Focus on the key information presented in the figure. "
            "Describe what the figure depicts (e.g., a graph, a diagram, a micrograph). "
            "If it's a graph, mention the axes and the general trend or comparison shown. "
            "If it's a diagram, explain what process or system it illustrates. "
            "Be concise and informative, as if you were writing a caption for this figure."
        )
        user_prompt_message = "Please describe the content of this scientific figure."
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": [
                    {"type": "text", "text": user_prompt_message},
                    {"type": "image_url", "image_url": {"url": image_data_url}},
                ]},
            ],
            max_tokens=500,
            temperature=0.3
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"An error occurred while generating image description for {image_path}: {e}")
        return None


# --- Main Processing Logic (UPDATED) ---
def process_markdown_images(markdown_content: str, markdown_dir: Path) -> str:
    """
    Finds all Markdown image links and replaces them with descriptions.
    Resolves image paths relative to the markdown file's directory.
    """
    image_pattern = r"!\[(.*?)\]\((.*?)\)"

    def replace_with_description(match: re.Match) -> str:
        original_image_link = match.group(0)
        relative_image_path_str = match.group(2)

        # --- THIS IS THE KEY CHANGE ---
        # Combine the markdown file's directory with the relative image path
        # to get a full, correct path to the image.
        absolute_image_path = markdown_dir.joinpath(relative_image_path_str)

        print(f"\nFound relative image path: {relative_image_path_str}")
        print(f"Resolving to full path: {absolute_image_path}")

        description = get_image_description_openai(absolute_image_path)

        if description:
            print(f"-> Success! Replacing image with description.")
            # We add newlines for better formatting in the final markdown
            im_list.append(relative_image_path_str)
            return f"\n\n**Image Description:** {description}\n*image_path: {relative_image_path_str}*\n"
        else:
            print(f"-> Failed to get description. Keeping original image link.")
            return original_image_link

    updated_content = re.sub(image_pattern, replace_with_description, markdown_content)
    return updated_content

if __name__ == "__main__":
    file_path = "scratch/29173_h00-with-image-refs.md"
    new_file_path = "scratch/29173_h00_with_descriptions.md"

    try:
        # Get the directory of the markdown file to resolve relative paths
        markdown_file_path = Path(file_path)
        markdown_dir = markdown_file_path.parent

        with open(markdown_file_path, 'r', encoding='utf-8') as f:
            original_md = f.read()

        print("--- Starting Markdown Processing ---")
        # Pass the directory to the processing function
        new_md = process_markdown_images(original_md, markdown_dir)
        print("\n--- Processing Complete ---")

        with open(new_file_path, 'w', encoding='utf-8') as f:
            f.write(new_md)
        print(f"\nSuccessfully created '{new_file_path}'.")

    except FileNotFoundError:
        print(f"Error: The input markdown file '{file_path}' was not found.")
        print("Please ensure the script is in the same directory as the markdown file, or provide a full path.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [0]:
im_list

In [0]:
import re
from pathlib import Path

# --- Heading Adjustment Function ---
def adjust_heading_levels(markdown_text: str) -> str:
    """
    Adjusts the heading levels in the markdown text.
    Changes top-level section headings (e.g., "## 3 ...") to H1 ("# 3 ...").
    Leaves subsections (e.g., "## 3.1 ...") alone.
    """
    def heading_replacer(match: re.Match) -> str:
        """Helper function for the substitution."""
        original_heading = match.group(0)
        number_part = match.group(2) # The number, e.g., "3" or "3.1"

        # Check if the number part contains a dot.
        # If it does, it's a subsection, so we don't change it.
        if '.' in number_part:
            return original_heading
        else:
            # It's a top-level section, so make it an H1 heading.
            title_part = match.group(3)
            return f"# {number_part}{title_part}"

    # This pattern finds any line starting with hashes and a section number.
    # It captures: 1) the hashes, 2) the number, 3) the rest of the line.
    heading_pattern = r"^(#+)\s+([\d\.]+)(.*)$"
    
    return re.sub(heading_pattern, heading_replacer, markdown_text, flags=re.MULTILINE)

# --- Execution Logic for this Cell ---
if __name__ == "__main__":
    # Define the input file (created by the previous cell) and the final output file.
    input_file_path = Path("scratch/29173_h00_with_descriptions.md")
    final_output_path = Path("scratch/29173_h00.md")

    try:
        print(f"Reading content from '{input_file_path}'...")
        # Read the content processed by the previous step.
        md_with_descriptions = input_file_path.read_text(encoding='utf-8')

        print("Adjusting heading levels for top-level sections...")
        # Apply the heading adjustment.
        final_md = adjust_heading_levels(md_with_descriptions)

        print(f"Saving final version to '{final_output_path}'...")
        # Write the result to a new file.
        final_output_path.write_text(final_md, encoding='utf-8')
        
        print("\nProcessing complete. Final file created successfully!")

    except FileNotFoundError:
        print(f"Error: The input file '{input_file_path}' was not found.")
        print("Please ensure the first part of the notebook ran successfully and created this file.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [0]:
# from langchain_text_splitters import MarkdownHeaderTextSplitter
# import markdown

# f = open("scratch/29173_h00_final_version.md", "r", encoding='utf-8')
# markdown_text = f.read()
# f.close()


# headers_to_split_on = [
#     ("#", "Header 1"),
# #     ("##", "Header 2"),
# #     ("###", "Header 3"),
# ]

# markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
# md_header_splits = markdown_splitter.split_text(markdown_text)
# md_header_splits

In [0]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
import os

# 1. Store the file path in a variable for easy reuse
file_path = "scratch/29173_h00.md"

# It's good practice to check if the file exists
if not os.path.exists(file_path):
    print(f"Error: File not found at {file_path}")
    # Create a dummy file for demonstration if it doesn't exist
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, "w", encoding='utf-8') as f:
        f.write("# Introduction\n\nThis is the first section.\n\n# Main Body\n\nThis is the main content.\n\n# Conclusion\n\nThis is the final part.")
        print(f"Created a dummy file at '{file_path}' for demonstration.")


with open(file_path, "r", encoding='utf-8') as f:
    markdown_text = f.read()

headers_to_split_on = [
    ("#", "Header 1"),
    #("##", "Header 2"),
    # ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_text)

# --- This is the new part ---
# 2. Iterate through the list of documents and update the metadata
for doc in md_header_splits:
    # 'source' is a common key used for the filename/source URI
    doc.metadata["source"] = pdf
    doc.metadata["images"] = []
    doc.metadata["page"] = ''
# -----------------------------


# 3. (Optional) Verify the result by printing the first chunk
if md_header_splits:
    #print("--- Content of the first chunk ---")
    print(md_header_splits[1].page_content)
    #print("\n--- Metadata of the first chunk ---")
    print(md_header_splits[1].metadata)
else:
    print("No chunks were created.")

In [0]:
md_header_splits[0]

In [0]:
headers = []
for i in md_header_splits:
    headers.append(i.metadata['Header 1'])
headers

In [0]:
import fitz  # PyMuPDF is imported as fitz
import re

# --- Configuration ---
HEADERS = [
    '3GPP TS 29.173 V17.0.0 (2022-04)',
    '3GPP',
    '1 Scope',
    '2 References',
    '3 Definitions, symbols and abbreviations',
    '4 General Description',
    '5 Diameter-based SLh Interface',
    '6 Protocol Specification and Implementations'
]
PDF_PATH = '29173_h00.pdf'

def find_header_page_pymupdf(doc, header_text: str) -> int | None:
    """
    Finds the page number where a specific header text first appears using PyMuPDF.
    """
    # PyMuPDF extraction is much cleaner, so we often don't need complex regex.
    # We will search for the literal text, ignoring case.
    for page_num, page in enumerate(doc):
        # page.get_text() is the PyMuPDF equivalent of extract_text()
        text = page.get_text()
        # A simple case-insensitive find is usually enough.
        if header_text.lower() in text.lower():
            return page_num + 1 # Return 1-based page number
    return None

# --- Main Script ---
try:
    # Use fitz.open() to open the document
    with fitz.open(PDF_PATH) as doc:
        total_pages = doc.page_count
        print(f"Analyzing '{PDF_PATH}' ({total_pages} pages total) using PyMuPDF...\n")

        header_locations = {}
        for header in HEADERS:
            page_num = find_header_page_pymupdf(doc, header)
            if page_num:
                header_locations[header] = page_num
            else:
                print(f"WARNING: Could not find header '{header}' in the document.")

        if not header_locations:
            print("\nCould not find any of the specified headers. Exiting.")
            exit()

        print("\n--- Section Page Counts ---")
        sorted_headers = sorted(header_locations.keys(), key=lambda h: header_locations[h])

        for i, header in enumerate(sorted_headers):
            start_page = header_locations[header]
            
            if i + 1 < len(sorted_headers):
                next_header = sorted_headers[i+1]
                end_page = header_locations[next_header]
                page_count = end_page - start_page
                if page_count <= 0:
                    page_count = 1
                    page_range_str = f"Page {start_page}"
                else:
                    page_range_str = f"Pages {start_page} to {end_page - 1}"
            else:
                end_page = total_pages
                page_count = end_page - start_page + 1
                page_range_str = f"Pages {start_page} to {end_page}"
            
            page_plural = "page" if page_count == 1 else "pages"
            print(f"- '{header}': {page_count} {page_plural} long ({page_range_str})")

except FileNotFoundError:
    print(f"ERROR: The file was not found at '{PDF_PATH}'")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [0]:
from PyPDF2 import PdfReader
with open(pdf, 'rb') as file:
    pdf_reader = PdfReader(file)
    num_pages = len(pdf_reader.pages)

In [0]:
num_pages 

In [0]:
# Make sure to install PyPDF2: pip install PyPDF2
from PyPDF2 import PdfReader
import os

# --- Data Structures (for clarity) ---
# The script assumes you have a pre-populated list of objects
# that look something like this Document class.
# You do NOT need to add this class to your code; it's just an illustration.
class Document:
    def __init__(self, page_content, metadata):
        self.page_content = page_content
        self.metadata = metadata 
    def __repr__(self):
        return f"Document(metadata={self.metadata})"

# --- Core Functions ---

def find_header_page(header_text, pdf_reader):
    """
    Finds the first page number where a given header text appears.

    Args:
        header_text (str): The text of the header to search for.
        pdf_reader (PdfReader): The PyPDF2 reader object.

    Returns:
        int: The page number (1-indexed) where the header is found.
        None: If the header is not found in the PDF.
    """
    num_pages = len(pdf_reader.pages)
    for page_num in range(num_pages):
        page = pdf_reader.pages[page_num]
        try:
            text = page.extract_text()
            # Check if text extraction was successful and if the header is present
            if text and header_text in text:
                return page_num + 1  # Return 1-indexed page number
        except Exception:
            # Some pages might not be extractable, so we continue
            continue
    return None # Return None if the header was not found

def add_page_ranges_to_metadata(pdf_path, headers, md_header_splits):
    """
    Extracts page ranges for headers in a PDF and modifies the 'page' metadata.

    Args:
        pdf_path (str): The path to the PDF file.
        headers (list): A list of header strings to find.
        md_header_splits (list): A list of existing objects (like Documents) 
                                 with a .metadata dictionary that will be modified.
    """
    if not os.path.exists(pdf_path):
        print(f"Error: PDF file not found at '{pdf_path}'.")
        # Populate metadata with an error message
        for item in md_header_splits:
            item.metadata['page'] = 'PDF not found'
        return

    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PdfReader(file)
            num_pages = len(pdf_reader.pages)

            # Step 1: Find the page for each header and store it.
            # This is more efficient than searching the PDF repeatedly.
            header_page_locations = [find_header_page(h, pdf_reader) for h in headers]

            # Step 2: Iterate through the headers to build the page range strings.
            for i in range(len(headers)):
                start_page = header_page_locations[i]

                # Handle case where the current header was not found.
                if start_page is None:
                    md_header_splits[i].metadata['page'] = 'Header Not Found'
                    continue

                # For all headers except the last one.
                if i < len(headers) - 1:
                    # Find the start page of the *next* header to use as the end page.
                    end_page = header_page_locations[i+1]
                    if end_page is not None:
                        # The section ends on the page *before* the next header starts
                        # unless they are on the same page.
                        effective_end_page = end_page -1 if end_page > start_page else start_page
                        md_header_splits[i].metadata['page'] = f'{start_page}-{effective_end_page}'
                    else:
                        # If the next header isn't found, the section runs to the end.
                        md_header_splits[i].metadata['page'] = f'{start_page}-{num_pages}'
                # For the very last header, the range goes to the end of the document.
                else:
                    md_header_splits[i].metadata['page'] = f'{start_page}-{num_pages}'

    except Exception as e:
        print(f"An error occurred while processing the PDF: {e}")


# --- Example Usage ---
# IMPORTANT: You must provide your actual data here.
# The script now assumes 'headers' and 'md_header_splits' are already defined
# and populated with your data before this function is called.

# 1. Define the path to your PDF
pdf_path = pdf # <-- Replace with your actual PDF path
# 3. Make sure 'md_header_splits' is your list of Document objects
# For example:
# md_header_splits = [Document(page_content="...", metadata={'source': '...', 'page': ''}), ...]
# (You don't need to redefine it here if it already exists in your script's context)


if 'md_header_splits' in locals() and len(headers) == len(md_header_splits):
    print("Starting PDF processing...")
    # This function will now MODIFY your existing md_header_splits list.
    add_page_ranges_to_metadata(pdf_path, headers, md_header_splits)

    # Print the results to see the populated metadata
    print("\n--- Processing Complete ---")
    for i, split in enumerate(md_header_splits):
        print(f"Header '{headers[i]}': {split.metadata}")
else:
    print("Error: 'md_header_splits' is not defined or its length doesn't match the headers list.")



In [0]:
md_header_splits[0]

In [0]:
#Adding images and pages? into chunk metadata

for i in range(len(md_header_splits)):
    f = True
    for j in im_list:
        if j in md_header_splits[i].page_content:
            md_header_splits[i].metadata['images'].append(j)
            f = False
    if f:
        md_header_splits[i].metadata['images'].append(None)


In [0]:
# def find_pdfs(root_dir: str) -> List[Path]:
#     root = Path(root_dir)
#     # The rglob pattern is case-sensitive by default; to catch .PDF too, you could check suffix lowercased:
#     pdfs = [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() == '.pdf']
#     return pdfs


# t = []
# files = find_pdfs('./data/')

# for i in files:
#     FILE_PATH = str(i)  

#     #doc = PyMuPDFLoader(FILE_PATH, mode='page') # open a document
#     text = doc.load()

#     for i in range(len(text)):
#         text[i].metadata['page'] += 1
#     t.extend(text)

# load_dotenv()

# print(len(t))

# text_splitter = RecursiveCharacterTextSplitter(
#     # Set a really small chunk size, just to show.
#     chunk_size=1000,
#     chunk_overlap=20,
#     length_function=len,
#     is_separator_regex=False,
# )

# texts = text_splitter.split_documents(t)

# try:
#     os.environ['OPENAI_API_KEY'] = dbutils.secrets.get(scope='common', key="NetDoc_OPENAI_KEY")
# except NameError:
#     raise EnvironmentError("dbutils is not available. Ensure you're running this in a supported environment or provide the OpenAI API key manually.")

# TOP_K = 5

In [0]:
EMBED_MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"

embedding = HuggingFaceEmbeddings(model_name=EMBED_MODEL_ID, model_kwargs={'device': 'cuda'})

vector_store = FAISS.from_documents(md_header_splits, embedding)

In [0]:
vector_store.save_local("f")

In [0]:
vector_store1 = FAISS.load_local("f", embedding, allow_dangerous_deserialization=True)

In [0]:
retriever = vector_store1.as_retriever(search_kwargs={"k": 5})

In [0]:
keyword_retriever = BM25Retriever.from_documents(md_header_splits)
keyword_retriever.k =  5
ensemble_retriever = EnsembleRetriever(retrievers=[retriever,keyword_retriever],weights=[0.5, 0.5])


In [0]:
import pickle

keyword_retriever = BM25Retriever.from_documents(md_header_splits)


with open('new.pkl', "wb") as f:
    pickle.dump(keyword_retriever, f)

# with open('test.pkl', 'rb') as f:
#     keyword_retriever = pickle.load(f)
#     keyword_retriever.k = 5

In [0]:
user_prompt = "What are the steps according to the HSS diagram"

retr = ensemble_retriever.get_relevant_documents(user_prompt)

texts= [(d.page_content, d.metadata) for d in retr]

print(texts[0])

In [0]:
texts[0]

In [0]:

try:
    os.environ['OPENAI_API_KEY'] = dbutils.secrets.get(scope='common', key="NetDoc_OPENAI_KEY")
except NameError:
    raise EnvironmentError("dbutils is not available. Ensure you're running this in a supported environment or provide the OpenAI API key manually.")

llm = ChatOpenAI(model='gpt-4o-mini')

prompt = PromptTemplate.from_template("You are currently a part of a RAG. Using these files: {input}, answer the user's prompt: {prompt} with the at most precesion possible. The files you have received are always organized with the page content first and then then metadata, so be sure to associate the right documents with the right metadata. Keep your answers clear and with as little filler as possible. Be to the point! If you believe that the piece of information within the prompt is not found in the given document chunks, say so rather than making up information. At the begining of your answer, cite the prompt given to you. At the end of your answer, give the source of the document your found the information in as well as the page number. If you found information from multiple pages or multpible document, cite them all.")

chain = prompt | llm
print(chain.invoke(
    {
        "prompt": user_prompt,
        "input": texts
    }
).content)